In [ ]:
from nicl.euclid.skypatch_noise import measure_noise_in_circular_annuli
from nicl.euclid.measure import create_bkgsub_images
from nicl.euclid.mask import create_combined_nir_mask, create_vis_mask

from pathlib import Path

from astropy.io import fits
from astropy.nddata import CCDData

from nicl.main import configure_logging

In [ ]:
configure_logging(name="nicl.euclid.mask", level="DEBUG")
configure_logging(name="nicl.mask", level="DEBUG")

## Measuring the skypatch noise

In [ ]:
profileDir = Path("/home/ppztk1/Erosita/Outputs_Clusters/cluster0/autoprof_results/")
outputDir = Path("/home/ppztk1/Erosita/Outputs_Clusters/v1.0/Skypatch_Noise/")
dataDir = Path("/home/ppztk1/euclid_data/Q1_R1_clusters_v1.0/skypatch/")
tempDir = outputDir / "temp_cleaned"
tempDir.mkdir(exist_ok=True)

fields = ["EDFS", "EDFF"]
box_sizes = [450, 500, 550, 650, 750, 1000, 1800, 2350]
bands = ["H", "J", "Y", "YJH"]  # no VIS needed for parallel run for now

for field in fields:
    for box_size in box_sizes:
        import nicl.euclid.mask

        nicl.euclid.mask.NIR_STACK_BKG_BOX_SIZE = box_size
        bkg_boxsize = box_size
        filter_size = nicl.euclid.mask.NIR_STACK_BKG_FILTER_SIZE

        prefix = f"{field}_Skypatch_bs{bkg_boxsize}"
        print(f"\n=== Measuring noise in {field} with box size {bkg_boxsize} ===")

        image_paths = {
            "H": dataDir / f"EUC_NIR_W-STK_H-{field}_sky.fits",
            "J": dataDir / f"EUC_NIR_W-STK_J-{field}_sky.fits",
            "Y": dataDir / f"EUC_NIR_W-STK_Y-{field}_sky.fits",
            "VIS": dataDir / f"EUC_VIS_SWL-STK-{field}_sky.fits",
            "YJH": outputDir / f"{prefix}_NIR_YJH_COADDED.fits",
        }

        nir_mask_path = outputDir / f"{prefix}_NIR_measurement_mask.fits"
        vis_mask_path = outputDir / f"{prefix}_VIS_measurement_mask.fits"

        prof_path = profileDir / "cluster0_H_no_noise_nomask_H.prof"

        if any(band in bands for band in ["H", "J", "Y", "YJH"]):
            if not nir_mask_path.exists():
                create_combined_nir_mask(
                    str(image_paths["H"]),
                    str(image_paths["J"]),
                    str(image_paths["Y"]),
                    centre_pos=False,
                    label=prefix,
                    output_dir=str(outputDir),
                )
                print("Created NIR masks.")
            else:
                print("NIR masks already exist.")

        if "VIS" in bands:
            if not vis_mask_path.exists():
                create_vis_mask(
                    str(image_paths["VIS"]),
                    centre_pos=False,
                    label=prefix,
                    output_dir=str(outputDir),
                )
                print("Created VIS masks.")
            else:
                print("VIS masks already exist.")

        for band in bands:
            print(f"\n--- Processing {band} ---")

            if band not in image_paths or not image_paths[band].exists():
                print(f"Image for band {band} does not exist. Skipping.")
                continue

            mask_path = (
                nir_mask_path if band in ["H", "J", "Y", "YJH"] else vis_mask_path
            )

            cleaned_paths = create_bkgsub_images(
                image_paths={band: image_paths[band]},
                background_mask_path=mask_path,
                output_background_dir=outputDir,
                temp_cleaned_dir=tempDir,
                box_size=bkg_boxsize,
                filter_size=filter_size,
                label=f"{prefix}_{band}",
            )

            cleaned_image_path = cleaned_paths[band]

            measure_noise_in_circular_annuli(
                image_path=cleaned_image_path,
                object_mask_path=mask_path,
                profile_path=prof_path,
                hdul_index=0,
                pixelscale=0.3,
                num_points=500,  # we potentially can increase this to 1000
                output_path=outputDir,
                verbose=f"{prefix}_{band}",
                show_annuli=False,
                show_diagnostic_plots=True,
                save_diagnostics=True,
            )

            Path(cleaned_image_path).unlink()
            print(f"Removed temp file: {cleaned_image_path}")